## Zelf een NN maken zonder library

We gaan nu zelf een Neuraal NEtwerk implementeren zonder gebruik te maken van libraries voor Neurale Netwerken. Sklearn gebruiken om de mnsit data te laden en numpy gebruiken mag uiteraard wel.

### Data laden

Laad de mnist data zoals je het eerder hebt geladen. Split dedara op in een test/training set. Bepaal zelf op hoeveel je split.

Normaliseer de data op 0-1 ipv de pixelwaardes. Dit heb je ook al eerder gedaan. 

Doe een one hot encoding op de labels van de data.

In [7]:
import numpy as np
import math
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.preprocessing import OneHotEncoder

# sparse_output=False: resultaat na encoding is normale NumPy array
encoder = OneHotEncoder(sparse_output=False)

# Load mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()


# Normalize images
train_images = train_images.astype(np.float32) / 255.0
test_images = test_images.astype(np.float32) / 255.0


# One hot encode labels
train_labels = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels = encoder.fit_transform(test_labels.reshape(-1, 1))


### Activatie functies

Maak de volgende activatiefuncties: 

    def relu(waardes)

en 

    def softmax(waardes) 

Als het goed is kun je ze uit de vorige opdracht gebruiken. 

In [8]:
def relu(z):
    return max(0, z)

def softmax(z):
    exp_values = [math.e ** v for v in z]
    sum_exp_values = sum(exp_values)
    return [round(ev / sum_exp_values, 3) for ev in exp_values]

### Cross entropy loss

Maak een functie

    def cross_entropy(y_true, y_predicted)

functie om de grootte van loss te kunnen berekenen. We gaan de waarde van de loss gebruiken als validatie output. Het wordt dus aan het einde geprint.

In [13]:
y_true = np.array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0])
y_pred = np.array([0.01, 0.02, 0.01, 0.05, 0.03, 0.02, 0.04, 0.75, 0.05, 0.02])

def cross_entropy_1(y_true, y_predicted):
    return -np.sum(y_true * np.log(y_predicted))

def cross_entropy_2(true, predicted):
    N = predicted.shape[0]
    loss = -1 / N * np.sum(true * np.log(predicted) + (1 - true) * np.log(1 - predicted))
    return loss


ce_1 = cross_entropy_1(y_true, y_pred)
ce_2 = cross_entropy_2(y_true, y_pred)
print(ce_1)
print(ce_2)


0.2876820724517809
0.05422586568914071


### Initialiseer je NN 

Maak de volgende variabelen:

- Kies hoeveel input nodes je hebt. 
- En hoeveel hidden layer nodes? 
- En hoeveel output nodes? Leg uit waarom je dat hebt gekozen.


Maak voor nu 1 input layer en 1 hidden layer. JE mag dit later uitbreiden

Om de initiele gewichten een random waarde te geven kun je np.random.randn gebruiken. Deze geeft een normale verdeling met waarden die ongeveer liggen tussen -3 en +3.

Om grote getallen te voorkomen kun je deze weer kleiner maken door bijv *0.01 te doen, dan wordt de standaardafwijking 0.01

### Forward pass
Maak een forward propagation functie. We returnen ook tussenwaarden (cache) zodat we die voor backpropagation runnen gebruiken.

#### Stap 1

Maak functie def forward()
Als parameters hgebruik de volgende:
- inputdata X
- gewichtsmatrix W1 van de hidden layer nodes
- biasvector b1 van de hidden layer nodes
- gewichtsmatrix W2 van de output layer nodes
- biasvector b2 van de output layer nodes

Bereken eerst de invoer van de hidden layer. Vermenigvuldig de input met de gewichten en tel de bias erbij op.


Hint:

denk aan matrixvermenigvuldiging
de vorm moet kloppen: (samples, features) @ (features, hidden)

#### Stap 2

De uitkomst van stap 1 is de ruwe activatie van de hidden layer. Pas nu de door jou geiplementeerde ReLU-functie hierop toe.

#### Stap 3

Gebruik de output van de hidden layer als input voor de outputlaag. Vermenigvuldig met W2 en tel de bias b2 er bij op. Eigenlijk bijna hetzelfde als wat je in stap 1 hebt gedaan maar met andere dimensies.

#### Stap 4

De output van stap 3 zijn “scores” (logits), nog geen kansen. Zet deze scores om naar kansen met de softmax-functie die je eerder hebt gemaakt.

#### Stap 5


Voor de backpropagation stap heb je straks de tussenwaarden nodig.
Sla alle relevante variabelen op in één object (bijv. tuple)

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

#### Stap 6

return nu (bijvoorbeeld in een tuple vorm) de voorspelling en de object uit Stap 5

### Backward pass

We gaan enkele functies voorbereiden die we in de les hebben gehad om de backpropagation stap uit te voeren.

We beginnen met berekening van de gradient van loss.

Tip: Je kunt de formule uit de slides van deze week gebruiken. 

Maak nu een methode 
        
        def compute_output_gradient(final_prediction, correct_answers)
        
die de output (laag) gradient teruggeeft. Je kunt de formule hierboven gebruiken.


Maak nu methode 


    def compute_output_gradients(hidden_output, output_gradient)

die de gradienten van de output‑gewichten teruggeeft. Voeg aan de output van je methode ook de gradient van bias toe (als tuple bijv.). 

Tip: Je kunt de formule uit de slides van deze week gebruiken.


Maak de methode 

    def compute_hidden_gradient(output_gradient, hidden_to_output_weights)
    
Deze zegt hoe sterk elke hidden neuron bijdraagt aan de fout.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

Schrijf een python functie die afgeleide RELU berekent voor een getal:

    def relu_derivative(x)

Pas het aan zodat het voor een lijst van getallen werkt (eg met numpy)


Maak nu methode 

    def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data)

De functie moet de gradienten van de hidden-laag gewichten en bias teruggeven. Je hebt de afgeleide van de RELU nodig uit de vorige opgave, want alleen die gewichten doen mee.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

Maak nu de backward propagation nmethode die de bovenstaatde methodes zal gbruiken.

    def backward(y_true, cache, W2):

De parameter cache zal uit de forward pass komen en als het goed is de volgende waardes bevatten:

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

Die kun je dus daaruit unpacken.

Doe nu de volgende stappen achter elkaar:

- Bereken de output gradient (gebruik je eerder gemaakte methode)
- Bereken de output gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en output gradient)

- Bereken de hidden gradient (gebruik je eerder gemaakte methode)
- Bereken de hidden gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en hidden gradient)

return de output gradients en de hidden gradients bijvoorbeeld in de vorm van een tuple:

    return dW1, db1, dW2, db2

### De training loop. 

JE mag het volgende stukje code gebruiken om je netwerk te trainen. MAar je mag uiteraard ook zelf een andere opzet maken.


In [10]:
# bedenk een learning rate die je wilt gebruiken
lr = 0.1

for epoch in range(20):

    # forward pass
    voorspelling, cache = forward(X_train, W1, b1, W2, b2)
    
    # loss berekenen we wel, maar doen er verder niet echt wat mee behalve printen
    # y_train_oh staat voor voor one hot encoded
    loss = cross_entropy(y_train_oh, voorspelling)

    # backward pass
    dW1, db1, dW2, db2 = backward(y_train_oh, cache, W2)

    # update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 5 == 0:
        preds = np.argmax(voorspelling, axis=1)

        # voor accuracy gebruik je class labels, niet one-hot vectors
        acc = np.mean(preds == y_train)
        print(f"Epoch {epoch}, Loss: {loss:.4f}, Acc: {acc:.3f}")

NameError: name 'forward' is not defined

One hot encoding toepassen (op de labels)

### Experimenteer

Werkt het? Mooi, dan kun je nu zelf experimenteren met verschillende setup parameters en kijken wat het beste werkt.